# Quant Midterm Project(Linear)

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.model_selection import cross_validate, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split


In [2]:
# Step 1: Load the datasets: The raw data has been preprocessed(categories have been set into dummies) and saved as parquet files
price_df = pd.read_parquet('price_final_train.parquet')   # Housing prices dataset
rent_df  = pd.read_parquet('rent_final_train.parquet')    # Rental prices dataset
print(price_df.head())
print(rent_df.head())

          Price    lnPrice  decoration_精装  decoration_简装  decoration_毛坯  \
0  6.194049e+06  15.639100            1.0            0.0            0.0   
1  4.354153e+06  15.286641            1.0            0.0            0.0   
2  3.321992e+06  15.016075            0.0            1.0            0.0   
3  7.895656e+06  15.881823            1.0            0.0            0.0   
4  1.902960e+06  14.458921            1.0            0.0            0.0   

   decoration_其他  high_dummy  middle_dummy  low_dummy  basement_dummy  ...  \
0            0.0           0             1          0               0  ...   
1            0.0           0             0          0               0  ...   
2            0.0           0             0          1               0  ...   
3            0.0           0             0          0               0  ...   
4            0.0           0             1          0               0  ...   

   water_civil  water_commercial  heating_central  heating_self  \
0          1.

Data Cleaning

Winsorization for all numerical variables.

Log transformation for price and number variables.

In [3]:
#   Select the numerical variables
def get_real_numeric_features(df):
    numeric_cols = df.select_dtypes(include=np.number).columns
    real_numeric = []
    for col in numeric_cols:
        unique_vals = set(df[col].dropna().unique())
        if not (unique_vals <= {0, 1}):
            real_numeric.append(col)
    return real_numeric

real_numeric_features = get_real_numeric_features(price_df)
print(real_numeric_features)

log_cols = [
    'total_floor', 'area', 'room_count', 'hall_count',
    'building_age', 'household_total', 'building_total',
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots'
]
for col in log_cols:
    if col in price_df.columns:
        price_df[col] = np.log1p(price_df[col])


['Price', 'lnPrice', 'total_floor', 'area', 'room_count', 'hall_count', '梯数', '户数', 'building_age', 'household_total', 'building_total', 'greening_rate', 'plot_ratio', 'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots', 'city']


In [4]:
#   Select the numerical variables
def get_real_numeric_features(df):
    numeric_cols = df.select_dtypes(include=np.number).columns
    real_numeric = []
    for col in numeric_cols:
        unique_vals = set(df[col].dropna().unique())
        if not (unique_vals <= {0, 1}):
            real_numeric.append(col)
    return real_numeric

real_numeric_features = get_real_numeric_features(rent_df)
print(real_numeric_features)

log_cols_2 = [
    'total_floor', 'area', 'room_count', 'hall_count', 'lease_min_months', 'lease_max_months', 'lease_avg_months',
    'building_age', 'household_total', 'building_total',
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots'
]
for col in log_cols_2:
    if col in rent_df.columns:
        rent_df[col] = np.log1p(rent_df[col])

['lnPrice', 'total_floor', 'area', 'room_count', 'hall_count', 'lease_min_months', 'lease_max_months', 'lease_avg_months', 'building_age', 'household_total', 'building_total', 'greening_rate', 'plot_ratio', 'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots', 'city']


In [5]:
def winsorize_columns(df, cols, lower=0.01, upper=0.99):
    df_winsor = df.copy()
    for col in cols:
        if col in df_winsor.columns and np.issubdtype(df_winsor[col].dtype, np.number):
            q_low = df_winsor[col].quantile(lower)
            q_high = df_winsor[col].quantile(upper)
            df_winsor[col] = np.clip(df_winsor[col], q_low, q_high)
    return df_winsor


num_price = [
    'total_floor', 'area', 'room_count', 'hall_count',  '梯数', '户数',
    'building_age', 'household_total', 'building_total',  'greening_rate', 'plot_ratio',
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots'
]

num_rent = [
    'total_floor', 'area', 'room_count', 'hall_count', 'lease_min_months', 'lease_max_months', 'lease_avg_months',
    'building_age', 'household_total', 'building_total', 'greening_rate', 'plot_ratio',
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots'
]

price_df = winsorize_columns(price_df, num_price)
rent_df = winsorize_columns(rent_df, num_rent)



Variable Selection

The time period is different from the training set, so drop the time variables in prediction.

In [6]:
drop_cols = [
    'trans_before_2020', 'trans_2021Q1', 'trans_2021Q2', 'trans_2021Q3', 'trans_2021Q4',
    'trans_2022Q1', 'trans_2022Q2', 'trans_2022Q3', 'trans_2022Q4',
    'trans_2023Q1', 'trans_2023Q2', 'trans_2023Q3', 'trans_2023Q4',
    'trans_2024Q1', 'trans_2024Q2', 'trans_2024Q3', 'trans_2024Q4', 'trans_2025Q1'
]
price_df = price_df.drop(columns=drop_cols, errors='ignore')
rent_df  = rent_df.drop(columns=drop_cols, errors='ignore')
print(price_df)

               Price    lnPrice  decoration_精装  decoration_简装  decoration_毛坯  \
0       6.194049e+06  15.639100            1.0            0.0            0.0   
1       4.354153e+06  15.286641            1.0            0.0            0.0   
2       3.321992e+06  15.016075            0.0            1.0            0.0   
3       7.895656e+06  15.881823            1.0            0.0            0.0   
4       1.902960e+06  14.458921            1.0            0.0            0.0   
...              ...        ...            ...            ...            ...   
103866  7.903242e+05  13.580198            0.0            0.0            0.0   
103867  1.113952e+06  13.923425            0.0            0.0            1.0   
103868  7.432028e+05  13.518724            0.0            1.0            0.0   
103869  1.290376e+06  14.070444            1.0            0.0            0.0   
103870  1.236439e+06  14.027746            0.0            0.0            0.0   

        decoration_其他  high_dummy  midd

Divide the data into different categories according to the city_id. Implement the missing data according to the city and include the interaction of city and other variables.

Note that our data is processed individually for each city.

In [7]:
#    Summary statistics by city_id
# pd.set_option('display.max_rows', 50)
# pd.set_option('display.max_columns', 50)
# pd.set_option('display.width', 100)

def city_summary(df, city_col='city'):
    summary = df.groupby(city_col).agg(
        sample_count = ('city', 'size'),
        **{f'{col}_missing': (col, lambda x: x.isnull().sum()) for col in df.columns if col != city_col}
    )
    return summary

price_city_summary = city_summary(price_df, city_col='city')
print(price_city_summary)

rent_city_summary = city_summary(rent_df, city_col='city')
print(rent_city_summary)

      sample_count  Price_missing  lnPrice_missing  decoration_精装_missing  \
city                                                                        
0            16491              0                0                     98   
1             6437              0                0                      0   
2            24996              0                0                      0   
3            21472              0                0                     44   
4             4363              0                0                      0   
5             3582              0                0                     58   
6             2281              0                0                      0   
7             1184              0                0                      0   
8             5931              0                0                      8   
9             1323              0                0                      0   
10           15057              0                0                    372   

In [8]:
#   Missing data implementation by city.
def fillna_by_city(df, city_col, num_cols):
    df_filled = df.copy()
    bin_cols = [col for col in df.columns if set(df[col].dropna().unique()) <= {0, 1} and col not in num_cols]
    for city, group in df.groupby(city_col):
        idx = group.index

        for col in num_cols:
            if col in df.columns:
                median = group[col].median()
                df_filled.loc[idx, col] = group[col].fillna(median)

        for col in bin_cols:
            mode = group[col].mode()
            fill_val = mode.iloc[0] if not mode.empty else 0
            df_filled.loc[idx, col] = group[col].fillna(fill_val)
    return df_filled

price_df = fillna_by_city(price_df, 'city', num_price)
rent_df  = fillna_by_city(rent_df, 'city', num_rent)

C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\User\A

In [9]:
#   Summary the variables that are all missing in each city.
pd.set_option('display.max_rows', None)
pd.set_option("display.width",1000)

def city_all_missing_vars(df, city_col='city'):
    result = {}
    for city, group in df.groupby(city_col):
        all_missing = group.isnull().all()
        missing_vars = all_missing[all_missing].index.tolist()
        result[city] = missing_vars
    return pd.Series(result, name='all_missing_vars')

price_city_missing = city_all_missing_vars(price_df, city_col='city')
print(price_city_missing)

rent_city_missing = city_all_missing_vars(rent_df, city_col='city')
print(rent_city_missing)
#   City_9
print(price_city_missing.loc[9])
print(rent_city_missing.loc[9])

0                                                    []
1                                                    []
2                                                    []
3                       [building_age, heating_fee_avg]
4                                                    []
5                                        [building_age]
6                                                    []
7                                                    []
8                                                    []
9     [building_age, household_total, building_total...
10                                                   []
11                                                   []
Name: all_missing_vars, dtype: object
0                                                    []
1                                                    []
2                                                    []
3                       [building_age, heating_fee_avg]
4                                                    []
5         

Process the missing value in each city category, our policy is:

They are all numerical variables, so fill in the missing values with the median value of the whole panel.

However, I also notice that these variables would not be included in the regression, since they have no variance, so they are scaled to 0.

In [10]:
def fillna_with_panel_median(df, num_cols):
    df_filled = df.copy()
    for col in num_cols:
        if col in df.columns:
            median = df[col].median()
            df_filled[col] = df[col].fillna(median)
    return df_filled

price_df = fillna_with_panel_median(price_df, num_price)
rent_df  = fillna_with_panel_median(rent_df, num_rent)

Feature Scaling for lasso: standardize the features(only the numerical variables) and leave out the response variable(also area) and categorical variables.

Note that the scaling should be within each city.

In [11]:
def scale_within_city(df, cols, city_col='city'):
    df_scaled = df.copy()
    #  city
    for city, idx in df.groupby(city_col).groups.items():
        sub = df.loc[idx, cols]
        means = sub.mean()
        stds = sub.std(ddof=0)
        stds_safe = stds.replace(0, np.nan)
        scaled = (sub - means) / stds_safe
        scaled = scaled.fillna(0)
        df_scaled.loc[idx, cols] = scaled
    return df_scaled

scaler_price_col = [
    'total_floor', 'room_count', 'hall_count', '梯数', '户数',
    'building_age', 'household_total', 'building_total',
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots'
]

scaler_rent_col = [
    'total_floor', 'room_count', 'hall_count', 'lease_min_months', 'lease_max_months', 'lease_avg_months',
    'building_age', 'household_total', 'building_total',
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots'
]

price_df = scale_within_city(price_df, scaler_price_col, city_col='city')
rent_df  = scale_within_city(rent_df, scaler_rent_col, city_col='city')

use the city as the dummy to interact with other variables.

In [12]:
keep_cols = ['area', 'lnPrice', 'Price']
keep_cols_price = [c for c in keep_cols if c in price_df.columns]
df_keep_price = price_df[keep_cols_price].copy()

city_dummies_price = pd.get_dummies(
    price_df['city'],
    prefix='city',
    drop_first=True,  # avoid multicollinearity
    dtype=np.uint8
)

# specified columns for interaction
interaction_cols_price = [
    'building_age',
    'heating_fee_avg',
    'subway'
]
loop_cols = [col for col in price_df.columns if 'loop_' in col]
heating_cols = [col for col in price_df.columns if 'heating_' in col]
interaction_cols_price.extend(loop_cols)
interaction_cols_price.extend(heating_cols)

# Ensure we only use columns that actually exist in the DataFrame
interaction_cols_price = [col for col in interaction_cols_price if col in price_df.columns]

print("Features selected for interaction with city dummies:")
print(interaction_cols_price)

Features selected for interaction with city dummies:
['building_age', 'heating_fee_avg', 'subway', 'loop_fifth_to_sixth', 'loop_fourth_to_fifth', 'loop_inner_ring_inside', 'loop_inner_to_middle', 'loop_inner_to_outer', 'loop_middle_to_outer', 'loop_outside_outer', 'loop_outside_sixth', 'loop_second_ring_inside', 'loop_second_to_third', 'loop_third_to_fourth', 'heating_central', 'heating_self', 'heating_fee_avg']


In [13]:
inter_parts_price = []
for col in interaction_cols_price:
    # Each column is broadcast-multiplied across all city dummies
    tmp = city_dummies_price.mul(price_df[col], axis=0)
    tmp.columns = [f"{col}__x__{cc}" for cc in tmp.columns]
    inter_parts_price.append(tmp)

inter_df_price = (
    pd.concat(inter_parts_price, axis=1)
    if inter_parts_price else
    pd.DataFrame(index=price_df.index)
)

In [14]:

#   Combine the final feature set
# 1. Get the main effect features (all columns except keep_cols and city)
main_effect_features = price_df.drop(columns=keep_cols_price + ['city'])

# 2. Assemble the final feature set correctly
price_df_final_features = pd.concat([
    df_keep_price,           # Contains area, Price, lnPrice
    main_effect_features,    # All other original features (as main effects)
    city_dummies_price,      # City dummies (as main effects / intercept shifts)
    inter_df_price           # The interaction terms we just created (slope shifts)
], axis=1)

print("\nShape of the original DataFrame:", price_df.shape)
print("Shape of the final DataFrame with interactions:", price_df_final_features.shape)



Shape of the original DataFrame: (103871, 74)
Shape of the final DataFrame with interactions: (103871, 271)


In [15]:

keep_cols = ['area', 'lnPrice']
keep_cols_rent = [c for c in keep_cols if c in rent_df.columns]
df_keep_rent = rent_df[keep_cols_rent].copy()

# 2. Create one-hot encoded city dummies for the rent data
city_dummies_rent = pd.get_dummies(
    rent_df['city'],
    prefix='city',
    drop_first=True,
    dtype=np.uint8
)

interaction_cols_rent = [
    'building_age',
    'heating_fee_avg',
    'lease_avg_months' # A rent-specific feature that could vary by city market dynamics
]
loop_cols_rent = [col for col in rent_df.columns if 'loop_' in col]
heating_cols_rent = [col for col in rent_df.columns if 'heating_' in col]
interaction_cols_rent.extend(loop_cols_rent)
interaction_cols_rent.extend(heating_cols_rent)


interaction_cols_rent = [col for col in interaction_cols_rent if col in rent_df.columns]

print("Features selected for interaction with city dummies (Rent Model):")
print(interaction_cols_rent)


inter_parts_rent = []
for col in interaction_cols_rent:
    tmp = city_dummies_rent.mul(rent_df[col], axis=0)
    tmp.columns = [f"{col}__x__{cc}" for cc in tmp.columns]
    inter_parts_rent.append(tmp)

inter_df_rent = (
    pd.concat(inter_parts_rent, axis=1)
    if inter_parts_rent else
    pd.DataFrame(index=rent_df.index)
)

main_effect_features_rent = rent_df.drop(columns=keep_cols_rent + ['city'])

rent_df_final_features = pd.concat([
    df_keep_rent,               # Contains area, Price, lnPrice
    main_effect_features_rent,  # All other original features
    city_dummies_rent,          # City dummies (for intercept shifts)
    inter_df_rent               # The new interaction terms (for slope shifts)
], axis=1)

print("\nShape of the original Rent DataFrame:", rent_df.shape)
print("Shape of the final Rent DataFrame with interactions:", rent_df_final_features.shape)


Features selected for interaction with city dummies (Rent Model):
['building_age', 'heating_fee_avg', 'lease_avg_months', 'loop_fifth_to_sixth', 'loop_fourth_to_fifth', 'loop_inner_ring_inside', 'loop_inner_to_middle', 'loop_inner_to_outer', 'loop_middle_to_outer', 'loop_outside_outer', 'loop_outside_sixth', 'loop_second_ring_inside', 'loop_second_to_third', 'loop_third_to_fourth', 'heating_self', 'heating_fee_avg']

Shape of the original Rent DataFrame: (98899, 66)
Shape of the final Rent DataFrame with interactions: (98899, 252)


Split the training data and testing data

We use the price/area to predict the price.


In [16]:
price_df_final_features['price_per_area'] = price_df_final_features['Price'] / price_df_final_features['area']
price_df_final_features['log_price_per_area'] = np.log1p(price_df_final_features['Price'] / price_df_final_features['area'])
rent_df_final_features['lnarea'] = np.log1p(rent_df_final_features['area'])
rent_df_final_features['log_price_per_area'] = rent_df_final_features['lnPrice'] - rent_df_final_features['lnarea']

In [17]:
# Prepare features (X) and target (y) for housing prices
X_price = price_df_final_features.drop(columns=['Price', 'lnPrice', 'price_per_area', 'log_price_per_area', 'area'])
y_price = price_df_final_features['log_price_per_area']

In [18]:
# Split housing data into train/test (80%/20%)
X_train_price, X_test_price, y_train_price, y_test_price = train_test_split(
    X_price, y_price, test_size=0.20, random_state=111
)

In [19]:
X_rent = rent_df_final_features.drop(columns=['lnPrice', 'log_price_per_area','lnarea', 'area'] )
y_rent = rent_df_final_features['log_price_per_area']

# Split rental data into train/test (80%/20%)
X_train_rent, X_test_rent, y_train_rent, y_test_rent = train_test_split(
    X_rent, y_rent, test_size=0.20, random_state=111
)

# PART 4: MODELING

In [20]:
## The modeling process for PRICE data

In [21]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.model_selection import cross_validate, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [22]:
# Define the models to be evaluated
models = {
    "OLS": LinearRegression(),
    "Lasso": Lasso(random_state=111, max_iter=10000),
    "Ridge": Ridge(random_state=111, max_iter=10000)
}
# List to store results for the final table
results_summary = []

In [23]:
original_price_df = pd.read_parquet('price_final_train.parquet')
original_rent_df = pd.read_parquet('rent_final_train.parquet')

print("--- Sample and Prediction Count Report (Price Model) ---")
print(f"Number of training samples used for fitting: {len(X_train_price)}")
print(f"Total PRICE predictions to be made on the test set: {len(X_test_price)}")
print("-" * 50)

--- Sample and Prediction Count Report (Price Model) ---
Number of training samples used for fitting: 83096
Total PRICE predictions to be made on the test set: 20775
--------------------------------------------------


In [24]:
results_summary_price = []
# %% md#### OLS (Ordinary Least Squares) Model
# 1. Initialize and fit the model
ols_model = LinearRegression()
ols_model.fit(X_train_price, y_train_price)
print(ols_model.get_params())
print("coef_:", ols_model.coef_)
print("intercept_:", ols_model.intercept_)
print("Train R2:", ols_model.score(X_train_price, y_train_price))
print("Test R2:", ols_model.score(X_test_price, y_test_price))

{'copy_X': True, 'fit_intercept': True, 'n_jobs': None, 'positive': False, 'tol': 1e-06}
coef_: [ 4.97296915e-01  4.03110566e-01  3.82473768e-01  4.46555540e-01
 -6.22311730e-03  2.05208483e-02  3.48079317e-02 -1.44952405e-01
 -7.82090387e-04  9.66288324e-02  1.88976220e-02  2.13564270e-01
  3.12783639e-02  1.31064784e-02  2.34812159e-02  4.03616275e-02
 -6.87418628e-02  1.22776822e-01  5.47160876e-01  1.24177347e-01
  4.94997119e-01  2.85001051e-01  3.07063253e-02 -3.07063254e-02
 -6.43129952e-02 -8.73122146e-02  9.60918258e-01  4.02249790e-01
 -1.21154284e+00  7.15255234e-04  4.37040953e-02 -2.03767426e-03
  6.30347880e-02  3.16577068e-02  3.87835171e-01  2.94413554e-01
  3.08516970e-01 -5.09031162e-02  1.10994422e-01 -3.10920393e-01
 -2.96853865e-01  9.00938534e-01  5.35044249e-01  4.26825331e-01
  7.71138548e-02 -4.92295333e-02  1.28926113e-02  4.67991112e+01
  4.31957890e-02 -6.68712982e-02 -1.63422520e-01  2.06062998e-02
  7.83212102e-03  2.03053576e-01  8.58193715e-02 -8.7017549

In [28]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1)
y_pred_train_log = ols_model.predict(X_train_price)  #  log1p(price/area)
area_train = price_df_final_features.loc[X_train_price.index, 'area']
y_true_train_price = price_df_final_features.loc[X_train_price.index, 'Price']

# price_per_area = (exp(y) - 1) * area = Price
y_pred_train_price = np.expm1(y_pred_train_log) * area_train
y_pred_train_price = np.maximum(y_pred_train_price, 0)  # Ensure no negative prices

in_sample_mae_ols = mean_absolute_error(y_true_train_price, y_pred_train_price)
in_sample_rmse_ols = np.sqrt(mean_squared_error(y_true_train_price, y_pred_train_price))
print(f"OLS In-sample MAE: {in_sample_mae_ols:,.2f} | RMSE: {in_sample_rmse_ols:,.2f}")

# 2) Test set evaluation
y_pred_test_log = ols_model.predict(X_test_price)
area_test = price_df_final_features.loc[X_test_price.index, 'area']
y_true_test_price = price_df_final_features.loc[X_test_price.index, 'Price']

y_pred_test_price = np.expm1(y_pred_test_log) * area_test
y_pred_test_price = np.maximum(y_pred_test_price, 0)

out_of_sample_mae_ols = mean_absolute_error(y_true_test_price, y_pred_test_price)
out_of_sample_rmse_ols = np.sqrt(mean_squared_error(y_true_test_price, y_pred_test_price))
print(f"OLS Out-of-sample MAE: {out_of_sample_mae_ols:,.2f} | RMSE: {out_of_sample_rmse_ols:,.2f}")

# 3) Cross-validation: 6-fold CV
cv_pred_log = cross_val_predict(ols_model, X_train_price, y_train_price, cv=6, n_jobs=-1)
cv_pred_price = np.expm1(cv_pred_log) * price_df_final_features.loc[X_train_price.index, 'area']
cv_pred_price = np.maximum(cv_pred_price, 0)
cv_true_price = price_df_final_features.loc[X_train_price.index, 'Price']
cv_mae_price = mean_absolute_error(cv_true_price, cv_pred_price)
print(f"OLS 6-fold CV MAE (price level): {cv_mae_price:.2f}")

# 4) Store results
results_summary_price.append({
    "Model": "OLS",
    "In-sample MAE": in_sample_mae_ols,
    "Out-of-sample MAE": out_of_sample_mae_ols,
    "Cross-validation MAE": cv_mae_price
})

OLS In-sample MAE: 687,461.21 | RMSE: 1,484,534.13
OLS Out-of-sample MAE: 672,238.27 | RMSE: 1,459,246.27
OLS 6-fold CV MAE (price level): 688618.69


In [35]:
# python
from sklearn.linear_model import LassoCV

# 1. Use LassoCV to automatically find the best alpha
# Since our y is small, the best alpha will likely be very small.
alphas_to_test = np.logspace(-6, 1, 100) # Test 100 alphas from 0.000001 to 10

# Initialize LassoCV. It performs cross-validation internally.
# NOTE: We are NOT using a Pipeline because scaling was already done correctly.
lasso_cv_model = LassoCV(
    alphas=alphas_to_test,
    cv=6,  # 6-fold cross-validation
    random_state=111,
    max_iter=10000,
    n_jobs=-1 # Use all available CPU cores
)

# 2. Fit LassoCV on the training data
print("Fitting LassoCV to find the best alpha...")
lasso_cv_model.fit(X_train_price, y_train_price)
print("Fitting complete.")
print(f"Best alpha found by LassoCV: {lasso_cv_model.alpha_}")

# The `lasso_cv_model` is now a fitted Lasso model using the BEST alpha.

# 3. Prepare area & true Price aligned with indices
area_train = price_df_final_features.loc[X_train_price.index, 'area']
area_test  = price_df_final_features.loc[X_test_price.index,  'area']
y_true_train_price = price_df_final_features.loc[X_train_price.index, 'Price']
y_true_test_price  = price_df_final_features.loc[X_test_price.index,  'Price']

# 4. In-sample evaluation
y_pred_train_log = lasso_cv_model.predict(X_train_price)
y_pred_train_price = np.expm1(y_pred_train_log) * area_train
y_pred_train_price = np.maximum(y_pred_train_price, 0)
in_sample_mae_lasso_cv = mean_absolute_error(y_true_train_price, y_pred_train_price)
in_sample_rmse_lasso_cv = np.sqrt(mean_squared_error(y_true_train_price, y_pred_train_price))
print(f"LassoCV In-sample MAE: {in_sample_mae_lasso_cv:,.2f} | RMSE: {in_sample_rmse_lasso_cv:,.2f}")

# 5. Out-of-sample evaluation
y_pred_test_log = lasso_cv_model.predict(X_test_price)
y_pred_test_price = np.expm1(y_pred_test_log) * area_test
y_pred_test_price = np.maximum(y_pred_test_price, 0)
out_of_sample_mae_lasso_cv = mean_absolute_error(y_true_test_price, y_pred_test_price)
out_of_sample_rmse_lasso_cv = np.sqrt(mean_squared_error(y_true_test_price, y_pred_test_price))
print(f"LassoCV Out-of-sample MAE: {out_of_sample_mae_lasso_cv:,.2f} | RMSE: {out_of_sample_rmse_lasso_cv:,.2f}")

# 6. CV MAE (using the predictions from the internal CV of LassoCV is tricky,
#    so we re-run cross_val_predict with the best model for a consistent result)
best_lasso_model = Lasso(alpha=lasso_cv_model.alpha_, random_state=111, max_iter=10000)
cv_pred_log = cross_val_predict(best_lasso_model, X_train_price, y_train_price, cv=6, n_jobs=-1)
cv_pred_price = np.expm1(cv_pred_log) * area_train
cv_pred_price = np.maximum(cv_pred_price, 0)
cv_mae_lasso_cv = mean_absolute_error(y_true_train_price, cv_pred_price)
cv_rmse_lasso_cv = np.sqrt(mean_squared_error(y_true_train_price, cv_pred_price))
print(f"Lasso (best alpha) 6-fold CV MAE (price level): {cv_mae_lasso_cv:.2f} | RMSE: {cv_rmse_lasso_cv:.2f}")

# 7. Store results
# Find and remove the old, bad Lasso result if it exists
results_summary_price = [res for res in results_summary_price if res.get("Model") != "Lasso"]
results_summary_price.append({
    "Model": "Lasso",
    "In-sample MAE": in_sample_mae_lasso_cv,
    "Out-of-sample MAE": out_of_sample_mae_lasso_cv,
    "Cross-validation MAE": cv_mae_lasso_cv
})

Fitting LassoCV to find the best alpha...


C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.376e+03, tolerance: 5.214e+00
  model = cd_fast.enet_coordinate_descent(


Fitting complete.
Best alpha found by LassoCV: 1e-06
LassoCV In-sample MAE: 687,509.15 | RMSE: 1,484,514.69
LassoCV Out-of-sample MAE: 672,358.92 | RMSE: 1,459,821.36
Lasso (best alpha) 6-fold CV MAE (price level): 688658.84 | RMSE: 1486778.35


In [59]:
import joblib
joblib.dump(lasso_cv_model, 'best_lasso_model.pkl')

模型已保存到 best_lasso_model.pkl 文件


In [36]:

import numpy as np
import pandas as pd
from pandas.api import types as ptypes
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Ensure results container exists
try:
    results_summary_price
except NameError:
    results_summary_price = []

# --- 0. Ensure feature dataframe exists
if 'price_df_final_features' not in globals():
    raise RuntimeError("`price_df_final_features` not found. Run feature engineering cells first.")

# --- 1. Normalize column names (as in your original code)
bad_labels = [c for c in price_df_final_features.columns if isinstance(c, (list, tuple, np.ndarray))]
if bad_labels:
    rename_map = {}
    for c in bad_labels:
        base = "_".join(map(str, c))
        new_name = base
        i = 1
        while new_name in price_df_final_features.columns:
            i += 1
            new_name = f"{base}_{i}"
        rename_map[c] = new_name
    price_df_final_features = price_df_final_features.rename(columns=rename_map)

# --- 2. Drop duplicated columns
price_df_final_features = price_df_final_features.loc[:, ~price_df_final_features.columns.duplicated()]

# --- 3. Ensure target exists
if 'log_price_per_area' not in price_df_final_features.columns:
    price_df_final_features['price_per_area'] = price_df_final_features['Price'] / price_df_final_features['area']
    price_df_final_features['log_price_per_area'] = np.log1p(price_df_final_features['price_per_area'])

# --- 4. Build X and y and re-split
X_price = price_df_final_features.drop(columns=['Price', 'lnPrice', 'price_per_area', 'log_price_per_area', 'area'], errors='ignore')
y_price = price_df_final_features['log_price_per_area']

X_train_price, X_test_price, y_train_price, y_test_price = train_test_split(
    X_price, y_price, test_size=0.20, random_state=111
)

# --- 5. Robustly identify continuous numeric columns to scale
continuous_cols = [c for c in X_train_price.columns if ptypes.is_numeric_dtype(X_train_price[c]) and X_train_price[c].nunique() > 2]

# --- 6. Build the preprocessing and Ridge pipeline (same as your code)
preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), continuous_cols)],
    remainder='passthrough'
)
ridge_pipeline = Pipeline([
    ('pre', preprocessor),
    ('ridge', Ridge(random_state=111, max_iter=10000))
])

# =============================================================================
# MODIFICATION: Use GridSearchCV to find the optimal alpha for Ridge
# The default alpha=1.0 is too strong and causes underfitting. We must find a better value.
# =============================================================================

# --- 7. Define the parameter grid to search for the best alpha
# We will search a range of values, from very small to moderately large.
param_grid = {
    'ridge__alpha': np.logspace(-4, 4, 10) # Test 10 alphas from 0.0001 to 10000
}

# --- 8. Set up and run GridSearchCV
print("Running GridSearchCV to find the best alpha for Ridge...")
ridge_grid_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=6,
    n_jobs=-1
)
ridge_grid_search.fit(X_train_price, y_train_price)
print("Fitting complete.")
print(f"Best alpha found by GridSearchCV for Ridge: {ridge_grid_search.best_params_['ridge__alpha']}")

# The `best_ridge_pipeline` is the one with the optimal alpha found by the search
best_ridge_pipeline = ridge_grid_search.best_estimator_

# --- 9. Prepare aligned area and true Price (same as your code)
area_train = price_df_final_features.loc[X_train_price.index, 'area']
area_test  = price_df_final_features.loc[X_test_price.index,  'area']
y_true_train_price = price_df_final_features.loc[X_train_price.index, 'Price']
y_true_test_price  = price_df_final_features.loc[X_test_price.index,  'Price']

# --- 10. In-sample evaluation using the BEST model
y_pred_train_log = best_ridge_pipeline.predict(X_train_price)
y_pred_train_price = np.expm1(y_pred_train_log) * area_train
y_pred_train_price = np.maximum(y_pred_train_price, 0)

in_sample_mae_ridge = mean_absolute_error(y_true_train_price, y_pred_train_price)
in_sample_rmse_ridge = np.sqrt(mean_squared_error(y_true_train_price, y_pred_train_price))
print(f"Ridge (best alpha) In-sample MAE: {in_sample_mae_ridge:,.2f} | RMSE: {in_sample_rmse_ridge:,.2f}")

# --- 11. Out-of-sample evaluation using the BEST model
y_pred_test_log = best_ridge_pipeline.predict(X_test_price)
y_pred_test_price = np.expm1(y_pred_test_log) * area_test
y_pred_test_price = np.maximum(y_pred_test_price, 0)

out_of_sample_mae_ridge = mean_absolute_error(y_true_test_price, y_pred_test_price)
out_of_sample_rmse_ridge = np.sqrt(mean_squared_error(y_true_test_price, y_pred_test_price))
print(f"Ridge (best alpha) Out-of-sample MAE: {out_of_sample_mae_ridge:,.2f} | RMSE: {out_of_sample_rmse_ridge:,.2f}")

# --- 12. 6-fold CV evaluation using the BEST model
cv_pred_log = cross_val_predict(best_ridge_pipeline, X_train_price, y_train_price, cv=6, n_jobs=-1)
cv_pred_price = np.expm1(cv_pred_log) * area_train
cv_pred_price = np.maximum(cv_pred_price, 0)

cv_mae_ridge = mean_absolute_error(y_true_train_price, cv_pred_price)
cv_rmse_ridge = np.sqrt(mean_squared_error(y_true_train_price, cv_pred_price))
print(f"Ridge (best alpha) 6-fold CV MAE (price level): {cv_mae_ridge:.2f} | RMSE: {cv_rmse_ridge:.2f}")

# --- 13. Store results
# First, remove the old Ridge result if it was appended previously
results_summary_price = [res for res in results_summary_price if res.get("Model") != "Ridge"]
results_summary_price.append({
    "Model": "Ridge",
    "In-sample MAE": in_sample_mae_ridge,
    "Out-of-sample MAE": out_of_sample_mae_ridge,
    "Cross-validation MAE": cv_mae_ridge
})

Running GridSearchCV to find the best alpha for Ridge...
Fitting complete.
Best alpha found by GridSearchCV for Ridge: 0.0001
Ridge (best alpha) In-sample MAE: 687,461.22 | RMSE: 1,484,534.01
Ridge (best alpha) Out-of-sample MAE: 672,238.30 | RMSE: 1,459,246.44
Ridge (best alpha) 6-fold CV MAE (price level): 688618.69 | RMSE: 1486797.60


In [37]:
# python
import numpy as np
import pandas as pd
from pandas.api import types as ptypes
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split, cross_val_predict, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Ensure results container exists
try:
    results_summary_price
except NameError:
    results_summary_price = []

# (Assume all previous data preparation steps and train_test_split have been run)
# ...

# =============================================================================
# MODIFICATION: We must use a Pipeline inside GridSearchCV
# This ensures that data scaling is performed correctly and independently
# for each fold of the cross-validation process, preventing data leakage
# and ensuring the hyperparameter search is valid.
# =============================================================================

# --- 1. Re-define the preprocessor and create the full pipeline
# (This is the same preprocessor from the corrected Ridge example)
continuous_cols = [c for c in X_train_price.columns if ptypes.is_numeric_dtype(X_train_price[c]) and X_train_price[c].nunique() > 2]

preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), continuous_cols)],
    remainder='passthrough'
)

elastic_pipeline = Pipeline([
    ('pre', preprocessor),
    ('regressor', ElasticNet(random_state=111, max_iter=10000))
])

# --- 2. Define the parameter grid for GridSearchCV
# CORRECTION: Parameter names MUST be prefixed with the pipeline step name (e.g., 'regressor__')
param_grid = {
    'regressor__alpha': [0.0001, 0.001, 0.01, 0.1],
    'regressor__l1_ratio': [0.1, 0.5, 0.9, 1.0] # 1.0 is pure Lasso, close to 0 is pure Ridge
}

# --- 3. Set up and run GridSearchCV on the PIPELINE
print("Running GridSearchCV for Best Linear Model (ElasticNet)...")
price_grid_search = GridSearchCV(
    estimator=elastic_pipeline, # We tune the entire pipeline
    param_grid=param_grid,
    cv=6,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)
price_grid_search.fit(X_train_price, y_train_price)
print(f"Best parameters found for ElasticNet Model: {price_grid_search.best_params_}")

# The best estimator is the entire pipeline with the optimal hyperparameters
best_price_pipeline = price_grid_search.best_estimator_

# --- 4. Prepare aligned area and true Price for evaluation
area_train = price_df_final_features.loc[X_train_price.index, 'area']
area_test  = price_df_final_features.loc[X_test_price.index,  'area']
y_true_train_price = price_df_final_features.loc[X_train_price.index, 'Price']
y_true_test_price  = price_df_final_features.loc[X_test_price.index,  'Price']

# --- 5. In-sample evaluation using the best found pipeline
y_pred_train_log = best_price_pipeline.predict(X_train_price)
y_pred_train_price = np.expm1(y_pred_train_log) * area_train
y_pred_train_price = np.maximum(y_pred_train_price, 0)
in_sample_mae_best = mean_absolute_error(y_true_train_price, y_pred_train_price)
in_sample_rmse_best = np.sqrt(mean_squared_error(y_true_train_price, y_pred_train_price))
print(f"Best Model In-sample MAE: {in_sample_mae_best:,.2f} | RMSE: {in_sample_rmse_best:,.2f}")

# --- 6. Out-of-sample evaluation using the best found pipeline
y_pred_test_log = best_price_pipeline.predict(X_test_price)
y_pred_test_price = np.expm1(y_pred_test_log) * area_test
y_pred_test_price = np.maximum(y_pred_test_price, 0)
out_of_sample_mae_best = mean_absolute_error(y_true_test_price, y_pred_test_price)
out_of_sample_rmse_best = np.sqrt(mean_squared_error(y_true_test_price, y_pred_test_price))
print(f"Best Model Out-of-sample MAE: {out_of_sample_mae_best:,.2f} | RMSE: {out_of_sample_rmse_best:,.2f}")

# --- 7. 6-fold CV evaluation using the best found pipeline for consistency
cv_pred_log = cross_val_predict(best_price_pipeline, X_train_price, y_train_price, cv=6, n_jobs=-1)
cv_pred_price = np.expm1(cv_pred_log) * area_train
cv_pred_price = np.maximum(cv_pred_price, 0)
cv_mae_best = mean_absolute_error(y_true_train_price, cv_pred_price)
cv_rmse_best = np.sqrt(mean_squared_error(y_true_train_price, cv_pred_price))
print(f"Best Model 6-fold CV MAE (price level): {cv_mae_best:.2f} | RMSE: {cv_rmse_best:.2f}")

# --- 8. Store results
# First, remove the old result if it was appended previously
results_summary_price = [res for res in results_summary_price if "Best Linear Model" not in res.get("Model")]
results_summary_price.append({
    "Model": "Best Linear Model (ElasticNet)",
    "In-sample MAE": in_sample_mae_best,
    "Out-of-sample MAE": out_of_sample_mae_best,
    "Cross-validation MAE": cv_mae_best
})

Running GridSearchCV for Best Linear Model (ElasticNet)...
Best parameters found for ElasticNet Model: {'regressor__alpha': 0.0001, 'regressor__l1_ratio': 0.1}
Best Model In-sample MAE: 688,874.87 | RMSE: 1,488,124.31
Best Model Out-of-sample MAE: 674,231.91 | RMSE: 1,474,792.29
Best Model 6-fold CV MAE (price level): 689974.85 | RMSE: 1490419.75


In [38]:

# %% md
#### Final Performance Summary Table (Price Model)
# %%
price_report_df = pd.DataFrame(results_summary_price)
price_final_report = price_report_df.rename(columns={
    "Model": "Metrics", "In-sample MAE": "In sample",
    "Out-of-sample MAE": "out of sample", "Cross-validation MAE": "Cross-validation"
}).set_index('Metrics')
price_final_report['Kaggle Score'] = "TBD"

pd.options.display.float_format = '{:,.2f}'.format
print("\n" + "="*60)
print("     FINAL PERFORMANCE SUMMARY (PRICE MODEL - MAE)     ")
print("="*60)
print(price_final_report)
print("="*60)


# =============================================================================
# NOTE: You would now repeat the ENTIRE process above for the RENT data.
# The logic is identical, but you must use the correct rent variables
# (X_train_rent, y_train_rent, etc.) and the specific back-transformation for rent.
# =============================================================================


     FINAL PERFORMANCE SUMMARY (PRICE MODEL - MAE)     
                                In sample  out of sample  Cross-validation Kaggle Score
Metrics                                                                                
Lasso                          687,509.15     672,358.92        688,658.84          TBD
Ridge                          687,461.22     672,238.30        688,618.69          TBD
Best Linear Model (ElasticNet) 688,874.87     674,231.91        689,974.85          TBD


Rent Modeling

In [33]:
ols_model_rent = LinearRegression()
ols_model_rent.fit(X_rent, y_rent)
print(ols_model_rent.get_params())
print("coef_:", ols_model_rent.coef_)
print("intercept_:", ols_model_rent.intercept_)
print("Train R2:", ols_model_rent.score(X_rent, y_rent))
print("Test R2:", ols_model_rent.score(X_rent, y_rent))

{'copy_X': True, 'fit_intercept': True, 'n_jobs': None, 'positive': False, 'tol': 1e-06}
coef_: [ 5.05683739e-02  7.67967319e-01  7.80130016e-01  7.88310797e-01
  7.83125569e-02  3.13445992e-02  1.46045496e-01  4.03028069e-02
  6.09588779e-04  6.64033953e-03  7.57227614e-13 -2.02304840e-12
 -4.75190720e-12  3.11402212e-02 -2.15638751e-02 -2.75113901e-02
 -3.88956876e-03  2.18246127e-02 -7.31093006e-01  1.06243501e-01
 -8.26354605e-02 -7.79678350e-02  1.16155210e-01  9.10280872e-02
  2.33973259e-02  7.99054964e-02 -1.22631642e-03 -9.32352155e-03
  3.23093251e-02  1.20149105e-02  2.82768334e-02  2.58640503e-02
  6.39668247e-02 -8.65277560e-03 -4.95254818e-02 -6.73887717e-02
  1.13973859e-02  6.48621375e-02  5.12934266e-02  9.11622782e-04
  3.06091139e-01  1.62327287e-01  8.45447321e-02 -1.97601906e-01
  8.62622117e-02 -2.75272458e-01 -3.41946576e-01  5.57646770e-01
  3.25859674e-01  3.59728572e-01  4.56084770e-02 -6.76623218e-02
  1.45106780e-02  4.84666952e+01  4.85823901e-02  1.1536207

In [ ]:
# python
import numpy as np
import pandas as pd
from pandas.api import types as ptypes
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Results container
results_summary_rent = []

# Ensure feature dataframe exists
if 'rent_df_final_features' not in globals():
    raise RuntimeError("`rent_df_final_features` not found. 请先运行特征工程单元。")

# 1) 规范非标量列名（tuple/list/ndarray）并去重列
bad_labels = [c for c in rent_df_final_features.columns if isinstance(c, (list, tuple, np.ndarray))]
if bad_labels:
    rename_map = {}
    for c in bad_labels:
        base = "_".join(map(str, c))
        new_name = base
        i = 1
        while new_name in rent_df_final_features.columns:
            i += 1
            new_name = f"{base}_{i}"
        rename_map[c] = new_name
    rent_df_final_features = rent_df_final_features.rename(columns=rename_map)

rent_df_final_features = rent_df_final_features.loc[:, ~rent_df_final_features.columns.duplicated()]

# 2) 确保目标列存在：`lnarea` 和 `log_price_per_area`
if 'lnarea' not in rent_df_final_features.columns:
    if 'area' not in rent_df_final_features.columns:
        raise RuntimeError("`area` 不存在于 `rent_df_final_features`。")
    rent_df_final_features['lnarea'] = np.log1p(rent_df_final_features['area'])

if 'log_price_per_area' not in rent_df_final_features.columns:
    if 'lnPrice' not in rent_df_final_features.columns:
        raise RuntimeError("`lnPrice` 不存在于 `rent_df_final_features`。")
    rent_df_final_features['log_price_per_area'] = rent_df_final_features['lnPrice'] - rent_df_final_features['lnarea']

# 3) 构建 X / y 并划分训练/测试
X_rent = rent_df_final_features.drop(columns=['lnPrice','log_price_per_area','lnarea','area'], errors='ignore')
y_rent = rent_df_final_features['log_price_per_area']

X_train_rent, X_test_rent, y_train_rent, y_test_rent = train_test_split(
    X_rent, y_rent, test_size=0.20, random_state=111
)

# 4) 识别需标准化的连续数值列
continuous_cols = [c for c in X_train_rent.columns if ptypes.is_numeric_dtype(X_train_rent[c]) and X_train_rent[c].nunique() > 2]

preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), continuous_cols)],
    remainder='passthrough'
)

# 小工具：从 `rent_df_final_features` 恢复真实的 Price（如果没有原始 Price 列，用 exp(lnPrice)）
def true_price_series(idx):
    if 'Price' in rent_df_final_features.columns:
        return rent_df_final_features.loc[idx, 'Price']
    return np.exp(rent_df_final_features.loc[idx, 'lnPrice'])

lnarea_train = rent_df_final_features.loc[X_train_rent.index, 'lnarea']
lnarea_test = rent_df_final_features.loc[X_test_rent.index, 'lnarea']
y_true_train_price = true_price_series(X_train_rent.index)
y_true_test_price = true_price_series(X_test_rent.index)

# 5) OLS (使用 Pipeline 确保同样预处理)
ols_pipeline = Pipeline([('pre', preprocessor), ('ols', LinearRegression())])
ols_pipeline.fit(X_train_rent, y_train_rent)

# 反变换并评估（rent: pred + lnarea -> lnPrice -> exp）
y_pred_train_log = ols_pipeline.predict(X_train_rent)
y_pred_train_lnPrice = y_pred_train_log + lnarea_train
y_pred_train_price = np.exp(y_pred_train_lnPrice)
y_pred_train_price = np.maximum(y_pred_train_price, 0)

in_sample_mae_ols = mean_absolute_error(y_true_train_price, y_pred_train_price)
in_sample_rmse_ols = np.sqrt(mean_squared_error(y_true_train_price, y_pred_train_price))

y_pred_test_log = ols_pipeline.predict(X_test_rent)
y_pred_test_lnPrice = y_pred_test_log + lnarea_test
y_pred_test_price = np.exp(y_pred_test_lnPrice)
y_pred_test_price = np.maximum(y_pred_test_price, 0)

out_of_sample_mae_ols = mean_absolute_error(y_true_test_price, y_pred_test_price)
out_of_sample_rmse_ols = np.sqrt(mean_squared_error(y_true_test_price, y_pred_test_price))

cv_pred_log = cross_val_predict(ols_pipeline, X_train_rent, y_train_rent, cv=6, n_jobs=-1)
cv_pred_price = np.exp(cv_pred_log + lnarea_train)
cv_pred_price = np.maximum(cv_pred_price, 0)
cv_mae_ols = mean_absolute_error(y_true_train_price, cv_pred_price)

results_summary_rent.append({
    "Model": "OLS",
    "In-sample MAE": in_sample_mae_ols,
    "Out-of-sample MAE": out_of_sample_mae_ols,
    "Cross-validation MAE": cv_mae_ols
})

# 6) Lasso: GridSearchCV on pipeline
lasso_pipeline = Pipeline([('pre', preprocessor), ('lasso', Lasso(max_iter=10000, random_state=111))])
lasso_grid = {'lasso__alpha': np.logspace(-6, 1, 30)}
lasso_search = GridSearchCV(lasso_pipeline, lasso_grid, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1)
lasso_search.fit(X_train_rent, y_train_rent)
best_lasso = lasso_search.best_estimator_

# 评估 Lasso
y_pred_train_log = best_lasso.predict(X_train_rent)
y_pred_train_price = np.exp(y_pred_train_log + lnarea_train)
y_pred_train_price = np.maximum(y_pred_train_price, 0)
in_sample_mae_lasso = mean_absolute_error(y_true_train_price, y_pred_train_price)

y_pred_test_log = best_lasso.predict(X_test_rent)
y_pred_test_price = np.exp(y_pred_test_log + lnarea_test)
y_pred_test_price = np.maximum(y_pred_test_price, 0)
out_of_sample_mae_lasso = mean_absolute_error(y_true_test_price, y_pred_test_price)

cv_pred_log = cross_val_predict(best_lasso, X_train_rent, y_train_rent, cv=6, n_jobs=-1)
cv_pred_price = np.exp(cv_pred_log + lnarea_train)
cv_pred_price = np.maximum(cv_pred_price, 0)
cv_mae_lasso = mean_absolute_error(y_true_train_price, cv_pred_price)

results_summary_rent.append({
    "Model": "Lasso",
    "In-sample MAE": in_sample_mae_lasso,
    "Out-of-sample MAE": out_of_sample_mae_lasso,
    "Cross-validation MAE": cv_mae_lasso
})

# 7) Ridge: GridSearchCV on pipeline
ridge_pipeline = Pipeline([('pre', preprocessor), ('ridge', Ridge(max_iter=10000, random_state=111))])
ridge_grid = {'ridge__alpha': np.logspace(-4, 4, 10)}
ridge_search = GridSearchCV(ridge_pipeline, ridge_grid, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1)
ridge_search.fit(X_train_rent, y_train_rent)
best_ridge = ridge_search.best_estimator_

y_pred_train_log = best_ridge.predict(X_train_rent)
y_pred_train_price = np.exp(y_pred_train_log + lnarea_train)
y_pred_train_price = np.maximum(y_pred_train_price, 0)
in_sample_mae_ridge = mean_absolute_error(y_true_train_price, y_pred_train_price)

y_pred_test_log = best_ridge.predict(X_test_rent)
y_pred_test_price = np.exp(y_pred_test_log + lnarea_test)
y_pred_test_price = np.maximum(y_pred_test_price, 0)
out_of_sample_mae_ridge = mean_absolute_error(y_true_test_price, y_pred_test_price)

cv_pred_log = cross_val_predict(best_ridge, X_train_rent, y_train_rent, cv=6, n_jobs=-1)
cv_pred_price = np.exp(cv_pred_log + lnarea_train)
cv_pred_price = np.maximum(cv_pred_price, 0)
cv_mae_ridge = mean_absolute_error(y_true_train_price, cv_pred_price)

results_summary_rent.append({
    "Model": "Ridge",
    "In-sample MAE": in_sample_mae_ridge,
    "Out-of-sample MAE": out_of_sample_mae_ridge,
    "Cross-validation MAE": cv_mae_ridge
})

# 8) ElasticNet: GridSearchCV on pipeline
elastic_pipeline = Pipeline([('pre', preprocessor), ('enet', ElasticNet(max_iter=10000, random_state=111))])
param_grid = {
    'enet__alpha': [1e-4, 1e-3, 1e-2, 1e-1],
    'enet__l1_ratio': [0.1, 0.5, 0.9, 1.0]
}
elastic_search = GridSearchCV(elastic_pipeline, param_grid, cv=6, scoring='neg_mean_absolute_error', n_jobs=-1)
elastic_search.fit(X_train_rent, y_train_rent)
best_enet = elastic_search.best_estimator_

y_pred_train_log = best_enet.predict(X_train_rent)
y_pred_train_price = np.exp(y_pred_train_log + lnarea_train)
y_pred_train_price = np.maximum(y_pred_train_price, 0)
in_sample_mae_enet = mean_absolute_error(y_true_train_price, y_pred_train_price)

y_pred_test_log = best_enet.predict(X_test_rent)
y_pred_test_price = np.exp(y_pred_test_log + lnarea_test)
y_pred_test_price = np.maximum(y_pred_test_price, 0)
out_of_sample_mae_enet = mean_absolute_error(y_true_test_price, y_pred_test_price)

cv_pred_log = cross_val_predict(best_enet, X_train_rent, y_train_rent, cv=6, n_jobs=-1)
cv_pred_price = np.exp(cv_pred_log + lnarea_train)
cv_pred_price = np.maximum(cv_pred_price, 0)
cv_mae_enet = mean_absolute_error(y_true_train_price, cv_pred_price)

results_summary_rent.append({
    "Model": "ElasticNet",
    "In-sample MAE": in_sample_mae_enet,
    "Out-of-sample MAE": out_of_sample_mae_enet,
    "Cross-validation MAE": cv_mae_enet
})

# 9) 汇总并展示
rent_report_df = pd.DataFrame(results_summary_rent)
rent_final_report = rent_report_df.rename(columns={
    "Model": "Metrics",
    "In-sample MAE": "In sample",
    "Out-of-sample MAE": "Out of sample",
    "Cross-validation MAE": "Cross-validation"
}).set_index('Metrics')
rent_final_report['Kaggle Score'] = "TBD"

pd.options.display.float_format = '{:,.2f}'.format
print("\nFINAL PERFORMANCE SUMMARY (RENT MODEL - MAE)")
print(rent_final_report)

FINAL PERFORMANCE SUMMARY (RENT MODEL - MAE)

| Model | In Sample | Out of Sample | Cross-Validation | Kaggle Score |
|-------|----------:|--------------:|-----------------:|-------------:|
| OLS   | 161,839.70 | 160,167.64   | 162,035.88      | TBD         |
| Lasso | 161,837.49 | 160,150.97   | 162,036.57      | TBD         |
| Ridge | 161,839.70 | 160,167.64   | 162,035.88      | TBD         |

In [60]:
import joblib
joblib.dump(best_lasso, 'best_lasso_rent.pkl')

['best_lasso_rent.pkl']

# Kaggle Test

In [25]:
# Step 1: Load the test datasets
price_test = pd.read_parquet('house_price_final_test.parquet')
rent_test = pd.read_parquet('rent_price_final_test.parquet')

print("Price test set shape:", price_test.shape)
print("Rent test set shape:", rent_test.shape)
print("Price test columns:", price_test.columns.tolist())
print("Rent test columns:", rent_test.columns.tolist())

# Save the ID columns for final submission
price_test_ids = price_test['ID'].copy()
rent_test_ids = rent_test['ID'].copy()

print(f"Price test IDs count: {len(price_test_ids)}")
print(f"Rent test IDs count: {len(rent_test_ids)}")


Price test set shape: (34017, 91)
Rent test set shape: (9773, 71)
Price test columns: ['ID', 'decoration_精装', 'decoration_简装', 'decoration_毛坯', 'decoration_其他', 'high_dummy', 'middle_dummy', 'low_dummy', 'basement_dummy', 'top_dummy', 'bottom_dummy', 'total_floor', 'area', 'room_count', 'hall_count', 'south_dummy', 'north_south_dummy', 'trans_before_2020', 'trans_2021Q1', 'trans_2021Q2', 'trans_2021Q3', 'trans_2021Q4', 'trans_2022Q1', 'trans_2022Q2', 'trans_2022Q3', 'trans_2022Q4', 'trans_2023Q1', 'trans_2023Q2', 'trans_2023Q3', 'trans_2023Q4', 'trans_2024Q1', 'trans_2024Q2', 'trans_2024Q3', 'trans_2024Q4', 'trans_2025Q1', '梯数', '户数', 'elevator_yes', 'villa_detached', 'villa_duplex', 'villa_semi_detached', 'villa_townhouse', 'transaction_commercial', 'transaction_non_commercial', 'usage_commercial_office', 'usage_commercial_residential', 'usage_high_end_residential', 'usage_ordinary_residential', 'usage_other_special', 'house_age_over_2_years', 'house_age_over_5_years', 'house_age_unde

In [26]:
# compare the column names of test sets with training sets
price_train_cols = set(price_df.columns)
rent_train_cols = set(rent_df.columns)
price_test_cols = set(price_test.columns)
rent_test_cols = set(rent_test.columns)
missing_price_cols = price_train_cols - price_test_cols
missing_rent_cols = rent_train_cols - rent_test_cols
print("Missing columns in price test set:", missing_price_cols)
print("Missing columns in rent test set:", missing_rent_cols)

Missing columns in price test set: {'lnPrice', 'Price'}
Missing columns in rent test set: {'lnPrice'}


In [27]:
log_cols = [
    'total_floor', 'area', 'room_count', 'hall_count',
    'building_age', 'household_total', 'building_total',
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots'
]
for col in log_cols:
    if col in price_test.columns:
        price_test[col] = np.log1p(price_test[col])

log_cols_2 = [
    'total_floor', 'area', 'room_count', 'hall_count', 'lease_min_months', 'lease_max_months', 'lease_avg_months',
    'building_age', 'household_total', 'building_total',
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots'
]
for col in log_cols_2:
    if col in rent_test.columns:
        rent_test[col] = np.log1p(rent_test[col])

num_price = [
    'total_floor', 'area', 'room_count', 'hall_count', '梯数', '户数',
    'building_age', 'household_total', 'building_total', 'greening_rate', 'plot_ratio',
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots'
]

num_rent = [
    'total_floor', 'area', 'room_count', 'hall_count', 'lease_min_months', 'lease_max_months', 'lease_avg_months',
    'building_age', 'household_total', 'building_total', 'greening_rate', 'plot_ratio',
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots'
]

price_test = winsorize_columns(price_test, num_price)
rent_test = winsorize_columns(rent_test, num_rent)

# 删除时间变量
drop_cols = [
    'trans_before_2020', 'trans_2021Q1', 'trans_2021Q2', 'trans_2021Q3', 'trans_2021Q4',
    'trans_2022Q1', 'trans_2022Q2', 'trans_2022Q3', 'trans_2022Q4',
    'trans_2023Q1', 'trans_2023Q2', 'trans_2023Q3', 'trans_2023Q4',
    'trans_2024Q1', 'trans_2024Q2', 'trans_2024Q3', 'trans_2024Q4', 'trans_2025Q1'
]
price_test = price_test.drop(columns=drop_cols, errors='ignore')
rent_test = rent_test.drop(columns=drop_cols, errors='ignore')

# 按城市填充缺失值
price_test = fillna_by_city(price_test, 'city', num_price)
rent_test = fillna_by_city(rent_test, 'city', num_rent)

# 用整个面板的中位数填充剩余的缺失值
price_test = fillna_with_panel_median(price_test, num_price)
rent_test = fillna_with_panel_median(rent_test, num_rent)

# 按城市标准化特征
scaler_price_col = [
    'total_floor', 'room_count', 'hall_count', '梯数', '户数',
    'building_age', 'household_total', 'building_total',
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots'
]

scaler_rent_col = [
    'total_floor', 'room_count', 'hall_count', 'lease_min_months', 'lease_max_months', 'lease_avg_months',
    'building_age', 'household_total', 'building_total',
    'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots'
]

price_test = scale_within_city(price_test, scaler_price_col, city_col='city')
rent_test = scale_within_city(rent_test, scaler_rent_col, city_col='city')

# 为价格测试集创建特征交互
keep_cols_price = ['area']  # 测试集没有Price和lnPrice
df_keep_price_test = price_test[keep_cols_price].copy()

city_dummies_price_test = pd.get_dummies(
    price_test['city'],
    prefix='city',
    drop_first=True,
    dtype=np.uint8
)

interaction_cols_price = [
    'building_age',
    'heating_fee_avg',
    'subway'
]
loop_cols = [col for col in price_test.columns if 'loop_' in col]
heating_cols = [col for col in price_test.columns if 'heating_' in col]
interaction_cols_price.extend(loop_cols)
interaction_cols_price.extend(heating_cols)
interaction_cols_price = [col for col in interaction_cols_price if col in price_test.columns]

inter_parts_price_test = []
for col in interaction_cols_price:
    tmp = city_dummies_price_test.mul(price_test[col], axis=0)
    tmp.columns = [f"{col}__x__{cc}" for cc in tmp.columns]
    inter_parts_price_test.append(tmp)

inter_df_price_test = (
    pd.concat(inter_parts_price_test, axis=1)
    if inter_parts_price_test else
    pd.DataFrame(index=price_test.index)
)

main_effect_features_price_test = price_test.drop(columns=keep_cols_price + ['city'])

price_test_final_features = pd.concat([
    df_keep_price_test,
    main_effect_features_price_test,
    city_dummies_price_test,
    inter_df_price_test
], axis=1)

# 为租金测试集创建特征交互
keep_cols_rent = ['area']
df_keep_rent_test = rent_test[keep_cols_rent].copy()

city_dummies_rent_test = pd.get_dummies(
    rent_test['city'],
    prefix='city',
    drop_first=True,
    dtype=np.uint8
)

interaction_cols_rent = [
    'building_age',
    'heating_fee_avg',
    'lease_avg_months'
]
loop_cols_rent = [col for col in rent_test.columns if 'loop_' in col]
heating_cols_rent = [col for col in rent_test.columns if 'heating_' in col]
interaction_cols_rent.extend(loop_cols_rent)
interaction_cols_rent.extend(heating_cols_rent)
interaction_cols_rent = [col for col in interaction_cols_rent if col in rent_test.columns]

inter_parts_rent_test = []
for col in interaction_cols_rent:
    tmp = city_dummies_rent_test.mul(rent_test[col], axis=0)
    tmp.columns = [f"{col}__x__{cc}" for cc in tmp.columns]
    inter_parts_rent_test.append(tmp)

inter_df_rent_test = (
    pd.concat(inter_parts_rent_test, axis=1)
    if inter_parts_rent_test else
    pd.DataFrame(index=rent_test.index)
)

main_effect_features_rent_test = rent_test.drop(columns=keep_cols_rent + ['city'])

rent_test_final_features = pd.concat([
    df_keep_rent_test,
    main_effect_features_rent_test,
    city_dummies_rent_test,
    inter_df_rent_test
], axis=1)

print("价格测试集最终特征形状:", price_test_final_features.shape)
print("租金测试集最终特征形状:", rent_test_final_features.shape)

C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\User\A

价格测试集最终特征形状: (34017, 270)
租金测试集最终特征形状: (9773, 252)


In [31]:
# Align Test Features with Training Features
def align_test_features(test_features, train_features):
    """Ensure test features have the same columns as training features"""
    # Get the columns from training features
    train_cols = train_features.columns

    # Find common columns between test and train
    common_cols = [col for col in train_cols if col in test_features.columns]
    missing_cols = [col for col in train_cols if col not in test_features.columns]
    extra_cols = [col for col in test_features.columns if col not in train_cols]

    print(f"训练集列数: {len(train_cols)}")
    print(f"测试集原始列数: {len(test_features.columns)}")
    print(f"共有列数: {len(common_cols)}")
    if missing_cols:
        print(f"测试集缺失的列 ({len(missing_cols)}个): {missing_cols}")
    if extra_cols:
        print(f"测试集多余的列将被删除 ({len(extra_cols)}个): {extra_cols}")

    # 检查是否有重复列名
    if len(test_features.columns) != len(set(test_features.columns)):
        duplicates = test_features.columns[test_features.columns.duplicated()].tolist()
        print(f"测试集中有重复列名: {duplicates}")
        # 删除重复列，保留第一个出现的
        test_features = test_features.loc[:, ~test_features.columns.duplicated()]

    if len(train_cols) != len(set(train_cols)):
        duplicates = train_cols[train_cols.duplicated()].tolist()
        print(f"训练集中有重复列名: {duplicates}")

    # Keep only the common columns in the same order as training features
    test_features_aligned = test_features[common_cols]

    # 最终验证
    print(f"最终对齐后测试集列数: {test_features_aligned.shape[1]}")

    return test_features_aligned

# Align test features with training features
price_test_features_aligned = align_test_features(price_test_final_features, X_price)
rent_test_features_aligned = align_test_features(rent_test_final_features, X_rent)

print("Aligned price test features shape:", price_test_features_aligned.shape)
print("Aligned rent test features shape:", rent_test_features_aligned.shape)

# 验证列数是否一致
print(f"价格训练集列数: {X_price.shape[1]}, 价格测试集列数: {price_test_features_aligned.shape[1]}")
print(f"租金训练集列数: {X_rent.shape[1]}, 租金测试集列数: {rent_test_features_aligned.shape[1]}")

训练集列数: 268
测试集原始列数: 270
共有列数: 268
测试集多余的列将被删除 (2个): ['area', 'ID']
测试集中有重复列名: ['heating_fee_avg__x__city_1', 'heating_fee_avg__x__city_2', 'heating_fee_avg__x__city_3', 'heating_fee_avg__x__city_4', 'heating_fee_avg__x__city_5', 'heating_fee_avg__x__city_6', 'heating_fee_avg__x__city_7', 'heating_fee_avg__x__city_8', 'heating_fee_avg__x__city_9', 'heating_fee_avg__x__city_10', 'heating_fee_avg__x__city_11']
训练集中有重复列名: ['heating_fee_avg__x__city_1', 'heating_fee_avg__x__city_2', 'heating_fee_avg__x__city_3', 'heating_fee_avg__x__city_4', 'heating_fee_avg__x__city_5', 'heating_fee_avg__x__city_6', 'heating_fee_avg__x__city_7', 'heating_fee_avg__x__city_8', 'heating_fee_avg__x__city_9', 'heating_fee_avg__x__city_10', 'heating_fee_avg__x__city_11']
最终对齐后测试集列数: 268
训练集列数: 250
测试集原始列数: 252
共有列数: 250
测试集多余的列将被删除 (2个): ['area', 'ID']
测试集中有重复列名: ['heating_fee_avg__x__city_1', 'heating_fee_avg__x__city_2', 'heating_fee_avg__x__city_3', 'heating_fee_avg__x__city_4', 'heating_fee_avg__x__city_5', 

In [34]:
# Make Predictions on Test Sets
# best_lasso_model 是房价预测模型，best_lasso 是租金预测模型

# 对测试集进行预测
price_log_pred = ols_model.predict(price_test_features_aligned)
rent_log_pred = ols_model_rent.predict(rent_test_features_aligned)

In [35]:
price_pred = np.expm1(price_log_pred)  # 如果使用的是 np.log1p，则用 expm1 还原
rent_pred = np.expm1(rent_log_pred)

# 乘以面积得到最终价格（假设模型预测的是单价）
if 'area' in price_test.columns:
    price_area = price_test['area']  # 直接使用原始面积
else:
    price_area = price_test_final_features['area']

if 'area' in rent_test.columns:
    rent_area = rent_test['area']  # 直接使用原始面积
else:
    rent_area = rent_test_final_features['area']

# 计算最终价格（单价 × 面积）
final_price_pred = price_pred * price_area
final_rent_pred = rent_pred * rent_area

# 确保预测值为非负
final_price_pred = np.maximum(final_price_pred, 0)
final_rent_pred = np.maximum(final_rent_pred, 0)

print(f"房价预测范围: {final_price_pred.min():.2f} - {final_price_pred.max():.2f}")
print(f"租金预测范围: {final_rent_pred.min():.2f} - {final_rent_pred.max():.2f}")

# 创建提交文件
price_submission = pd.DataFrame({
    'ID': price_test_ids,
    'Predicted': final_price_pred
})

rent_submission = pd.DataFrame({
    'ID': rent_test_ids,
    'Predicted': final_rent_pred
})

# 合并房价和租金预测
submission = pd.concat([price_submission, rent_submission], ignore_index=True)

# 检查提交文件
print(f"提交文件形状: {submission.shape}")
print(f"ID唯一值数量: {submission['ID'].nunique()}")
print(submission.head())

# 保存提交文件
submission.to_csv('submission.csv', index=False)
print("提交文件已保存为 submission.csv")

房价预测范围: 42341.25 - 36455183.11
租金预测范围: 34021.43 - 3264121.66
提交文件形状: (43790, 2)
ID唯一值数量: 43790
        ID     Predicted
0  1000000  1.109903e+07
1  1000001  3.904929e+06
2  1000002  4.796979e+06
3  1000003  2.316010e+06
4  1000004  1.028931e+07
提交文件已保存为 submission.csv
